# SBTi 데이터 분석

먼저 SBTi 데이터를 확인하면서 글로벌 기업들이 기후목표를 어떻게 설정하고 있는지 살펴보려고 한다.
처음에는 데이터 구조와 결측치 등을 확인하고, 이후 지역·산업·목표 상태를 중심으로 분석할 예정이다.

In [2]:
import pandas as pd

df_sbti = pd.read_csv("../data/sbti.csv")

df_sbti.head()

,sbti_id,company_name,isin,lei,organization_type,location,region,sector,near_term_status,near_term_target_classification,...,full_target_language,ba15_status,ba15_date,target_classification_long,reason_for_extension_or_removal,date_updated,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,40023788.0,"""Hytex Plastic"" CJSC",NaN,NaN,SME,Armenia,Europe,Containers and Packaging,Targets set,1.5°C,...,This target was approved using a streamlined t...,NaN,NaN,The targets covering greenhouse gas emissions ...,NaN,2025-09-18 01:00:00,NaN,NaN,NaN,NaN
1,40026740.0,"""Kazakh pharmaceutical company ""MEDSERVICE PL...",NaN,NaN,Corporate,Kazakhstan,Asia,"Healthcare Providers and Services, and Healthc...",Committed,NaN,...,NaN,NaN,NaN,NaN,NaN,2025-07-31 01:00:00,NaN,NaN,NaN,NaN
2,40009945.0,& Co.,NaN,NaN,SME,Denmark,Europe,Media,Targets set,1.5°C,...,This target was approved using a streamlined t...,NaN,NaN,The targets covering greenhouse gas emissions ...,NaN,2024-01-25 01:00:00,NaN,NaN,NaN,NaN
3,40022773.0,&Tradition,NaN,894500RY226L5NMCTT65,Corporate,Denmark,Europe,"Consumer Durables, Household and Personal Prod...",Targets set,1.5°C,...,Near-Term Targets: &Tradition A/S commits to r...,NaN,NaN,The targets covering greenhouse gas emissions ...,NaN,2025-07-10 01:00:00,NaN,NaN,NaN,NaN
4,40009210.0,(ACIP) Alexandria Company for Industrial Packages,NaN,NaN,SME,Egypt,Africa,Containers and Packaging,Targets set,Well-below 2°C,...,This target was approved using a streamlined t...,NaN,NaN,The targets covering greenhouse gas emissions ...,NaN,2022-02-03 01:00:00,NaN,NaN,NaN,NaN


In [3]:
df_sbti.columns

Index(['sbti_id', 'company_name', 'isin', 'lei', 'organization_type',
       'location', 'region', 'sector', 'near_term_status',
       'near_term_target_classification', 'near_term_target_year',
       'long_term_status', 'long_term_target_classification',
       'long_term_target_year', 'net_zero_status', 'net_zero_year',
       'full_target_language', 'ba15_status', 'ba15_date',
       'target_classification_long', 'reason_for_extension_or_removal',
       'date_updated', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25'],
      dtype='str')

In [4]:
df_sbti.shape

(15654, 26)

In [5]:
# Unnamed 열에 실제 값이 들어있는지 확인

df_sbti[['Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25']].isna().sum()

Unnamed: 22    15654
Unnamed: 23    15654
Unnamed: 24    15654
Unnamed: 25    15654
dtype: int64

In [6]:
# 값이 전부 비어 있는 Unnamed 열 삭제
df_sbti = df_sbti.drop(
    columns=['Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25']
)

# 삭제 후 데이터 크기 확인
df_sbti.shape

(15654, 22)

In [7]:
df_sbti.isna().sum()

sbti_id                                0
company_name                           0
isin                               12689
lei                                11460
organization_type                      0
location                               0
region                                 0
sector                                 0
near_term_status                       0
near_term_target_classification     3658
near_term_target_year               3658
long_term_status                   12902
long_term_target_classification    12902
long_term_target_year              12902
net_zero_status                    10599
net_zero_year                      12851
full_target_language                3657
ba15_status                        13499
ba15_date                          13499
target_classification_long          3657
reason_for_extension_or_removal    13499
date_updated                           0
dtype: int64

In [8]:
# Near-term 목표 상태에 따라 목표 분류 값이 비어 있는지 확인
pd.crosstab(
    df_sbti['near_term_status'],
    df_sbti['near_term_target_classification'].isna()
)

near_term_target_classification,False,True
near_term_status,,
Commitment removed,0,1442
Committed,0,2215
Targets set,11996,1


In [9]:
# Targets set 상태인데 목표 분류가 비어 있는 1개 기업 확인
df_sbti[
    (df_sbti['near_term_status'] == 'Targets set') &
    (df_sbti['near_term_target_classification'].isna())
]

,sbti_id,company_name,isin,lei,organization_type,location,region,sector,near_term_status,near_term_target_classification,...,long_term_target_classification,long_term_target_year,net_zero_status,net_zero_year,full_target_language,ba15_status,ba15_date,target_classification_long,reason_for_extension_or_removal,date_updated
12926,40030035.0,Stirling Square,NaN,NaN,Financial Institution,United Kingdom,Europe,"Banks, Diverse Financials, Insurance",Targets set,NaN,...,1.5°C,2050.0,Targets set,2050.0,Headline target: Stirling Square commits to re...,NaN,NaN,The targets covering greenhouse gas emissions ...,NaN,2026-06-11 01:00:00


In [10]:
# 전체 행에서 중복된 데이터가 있는지 확인
df_sbti.duplicated().sum()

np.int64(0)

In [11]:
# SBTi ID와 기업명 기준 중복 확인
print("SBTi ID 중복:", df_sbti['sbti_id'].duplicated().sum())
print("기업명 중복:", df_sbti['company_name'].duplicated().sum())

SBTi ID 중복: 0
기업명 중복: 0


In [12]:
# Near-term 목표연도가 어떤 형태로 들어있는지 확인
df_sbti['near_term_target_year'].value_counts(dropna=False).head(15)

near_term_target_year
2030.0    7390
NaN       3658
FY2030    1727
2035.0     633
2034.0     416
2032.0     277
2033.0     269
FY2034     206
2031.0     202
FY2035     199
FY2031     116
2029.0      95
FY2033      90
FY2032      78
2028.0      70
Name: count, dtype: int64

In [13]:
# Near-term 목표연도별 기업 수를 보기 쉽게 확인
year_count = (
    df_sbti['near_term_target_year']
    .value_counts(dropna=False)
    .head(15)
    .reset_index()
)

year_count.columns = ['목표연도', '기업수']

year_count

,목표연도,기업수
0,2030.0,7390
1,NaN,3658
2,FY2030,1727
3,2035.0,633
4,2034.0,416
5,2032.0,277
6,2033.0,269
7,FY2034,206
8,2031.0,202
9,FY2035,199


In [14]:
# 목표연도에서 숫자만 추출해서 분석용 연도 컬럼 생성
df_sbti['near_term_year_clean'] = (
    df_sbti['near_term_target_year']
    .astype(str)
    .str.extract(r'(\d{4})')[0]
)

# 숫자형으로 변환
df_sbti['near_term_year_clean'] = pd.to_numeric(
    df_sbti['near_term_year_clean'],
    errors='coerce'
)

# 정리된 연도 확인
df_sbti['near_term_year_clean'].value_counts(dropna=False).head(10)

near_term_year_clean
2030.0    9117
NaN       3658
2035.0     832
2034.0     622
2033.0     359
2032.0     355
2031.0     318
2029.0     124
2028.0      83
2027.0      61
Name: count, dtype: int64

In [15]:
# Near-term 목표 상태별 기업 수 확인
near_status = df_sbti['near_term_status'].value_counts()

near_status

near_term_status
Targets set           11997
Committed              2215
Commitment removed     1442
Name: count, dtype: int64

In [16]:
# Near-term 목표 상태별 비율 계산
near_status_pct = (
    df_sbti['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

near_status_pct

near_term_status
Targets set           76.6
Committed             14.1
Commitment removed     9.2
Name: proportion, dtype: float64

In [17]:
# Near-term 목표 상태별 기업 수와 비율을 하나의 표로 정리
near_status_table = pd.DataFrame({
    '기업수': df_sbti['near_term_status'].value_counts(),
    '비율(%)': df_sbti['near_term_status'].value_counts(normalize=True)
                .mul(100)
                .round(1)
})

near_status_table

,기업수,비율(%)
near_term_status,,
Targets set,11997,76.6
Committed,2215,14.1
Commitment removed,1442,9.2


In [18]:
import sys
import nbformat

print("Python 경로:", sys.executable)
print("nbformat 버전:", nbformat.__version__)

Python 경로: C:\work\PJ3\.venv\Scripts\python.exe
nbformat 버전: 5.11.1


In [19]:
import plotly.express as px

# 그래프에 사용하기 위해 인덱스를 컬럼으로 변경
near_status_plot = near_status_table.reset_index()
near_status_plot.columns = ['목표상태', '기업수', '비율(%)']

# Near-term 목표 상태별 기업 수 시각화
fig = px.bar(
    near_status_plot,
    x='목표상태',
    y='기업수',
    text='비율(%)',
    title='SBTi 기업의 Near-term 목표 상태'
)

# 막대 위에 비율 표시
fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.show()

### 분석 결과

SBTi에 등록된 기업들을 확인한 결과, 76.6%가 단기 감축목표를 이미 설정한 상태였다.
14.1%는 목표 설정을 약속한 단계였고, 9.2%는 기존 약속이 철회된 상태였다.

전체적으로 보면 SBTi에 참여한 기업들은 단순히 참여하는 데 그치기보다 실제 목표 설정 단계까지 진행한 경우가 많았다.

다만 이 결과는 전 세계 모든 기업을 대상으로 한 것이 아니라 SBTi 데이터에 등록된 기업들만을 기준으로 한 결과이기 때문에, 글로벌 기업 전체의 현황으로 확대해서 해석하지는 않았다.

In [20]:
# 단기 감축목표를 설정한 기업만 추출
target_set = df_sbti[
    df_sbti['near_term_status'] == 'Targets set'
]

# 목표 수준별 기업 수 확인
target_set['near_term_target_classification'].value_counts(dropna=False)# 목표 분류가 여러 개 같이 표시된 기업만 확인
multi_class = target_set[
    target_set['near_term_target_classification'].str.contains('/', na=False)
]

multi_class[
    [
        'company_name',
        'organization_type',
        'sector',
        'near_term_target_classification'
    ]
].head(30)

,company_name,organization_type,sector,near_term_target_classification
567,alstria office AG,Corporate,Real Estate,1.5°C/Well-below 2°C
1811,BMW Group,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
2490,China Motor Corporation,Corporate,Automobiles and Components,1.5°C/1.5°C
2522,"Chongqing Linglong Industrial Development co.,...",Corporate,Automobiles and Components,1.5°C/1.5°C
2814,Comer Industries,Corporate,Automobiles and Components,1.5°C/1.5°C
3848,ECARX Holdings inc.,Corporate,Automobiles and Components,1.5°C/1.5°C
4716,Ford Motor Company,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
5092,General Motors,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
5916,HES Präzisionsteile Hermann Erkert GmbH,Corporate,Automobiles and Components,1.5°C/1.5°C
6272,Hyundai Motor Company,Corporate,Automobiles and Components,1.5°C/1.5°C


In [21]:
# 목표 분류가 여러 개 같이 표시된 기업만 확인
multi_class = target_set[
    target_set['near_term_target_classification'].str.contains('/', na=False)
]

multi_class[
    [
        'company_name',
        'organization_type',
        'sector',
        'near_term_target_classification'
    ]
].head(30)

,company_name,organization_type,sector,near_term_target_classification
567,alstria office AG,Corporate,Real Estate,1.5°C/Well-below 2°C
1811,BMW Group,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
2490,China Motor Corporation,Corporate,Automobiles and Components,1.5°C/1.5°C
2522,"Chongqing Linglong Industrial Development co.,...",Corporate,Automobiles and Components,1.5°C/1.5°C
2814,Comer Industries,Corporate,Automobiles and Components,1.5°C/1.5°C
3848,ECARX Holdings inc.,Corporate,Automobiles and Components,1.5°C/1.5°C
4716,Ford Motor Company,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
5092,General Motors,Corporate,Automobiles and Components,1.5°C/Well-below 2°C
5916,HES Präzisionsteile Hermann Erkert GmbH,Corporate,Automobiles and Components,1.5°C/1.5°C
6272,Hyundai Motor Company,Corporate,Automobiles and Components,1.5°C/1.5°C


In [22]:
# 복수의 목표 분류가 있는 기업 중 대표 사례 확인
check_companies = [
    'alstria office AG',
    'BMW Group',
    'Hyundai Motor Company'
]

df_sbti.loc[
    df_sbti['company_name'].isin(check_companies),
    [
        'company_name',
        'near_term_target_classification',
        'full_target_language',
        'target_classification_long'
    ]
]

,company_name,near_term_target_classification,full_target_language,target_classification_long
567,alstria office AG,1.5°C/Well-below 2°C,Germany-based real estate company alstria offi...,The targets covering greenhouse gas emissions ...
1811,BMW Group,1.5°C/Well-below 2°C,BMW Group commits to reduce scope 1 and 2 GHG ...,The targets covering greenhouse gas emissions ...
6272,Hyundai Motor Company,1.5°C/1.5°C,Near-Term Targets:\nHyundai Motor Company comm...,The targets covering greenhouse gas emissions ...


In [23]:
# 복수 분류가 있는 기업들의 설명 문구 확인
multi_class[
    [
        'company_name',
        'near_term_target_classification',
        'target_classification_long'
    ]
]

,company_name,near_term_target_classification,target_classification_long
567,alstria office AG,1.5°C/Well-below 2°C,The targets covering greenhouse gas emissions ...
1811,BMW Group,1.5°C/Well-below 2°C,The targets covering greenhouse gas emissions ...
2490,China Motor Corporation,1.5°C/1.5°C,The targets covering greenhouse gas emissions ...
2522,"Chongqing Linglong Industrial Development co.,...",1.5°C/1.5°C,The targets covering greenhouse gas emissions ...
2814,Comer Industries,1.5°C/1.5°C,The targets covering greenhouse gas emissions ...
3848,ECARX Holdings inc.,1.5°C/1.5°C,The targets covering greenhouse gas emissions ...
4716,Ford Motor Company,1.5°C/Well-below 2°C,The targets covering greenhouse gas emissions ...
5092,General Motors,1.5°C/Well-below 2°C,The targets covering greenhouse gas emissions ...
5916,HES Präzisionsteile Hermann Erkert GmbH,1.5°C/1.5°C,The targets covering greenhouse gas emissions ...
6272,Hyundai Motor Company,1.5°C/1.5°C,The targets covering greenhouse gas emissions ...


In [24]:
# Near-term 목표 분류를 Scope 1·2와 Scope 3 Category 11로 분리
classification_split = (
    target_set['near_term_target_classification']
    .str.split('/', expand=True)
)

target_set = target_set.copy()

target_set['scope12_class'] = classification_split[0]
target_set['scope3_cat11_class'] = classification_split[1]

# 분리 결과 확인
target_set[
    [
        'company_name',
        'near_term_target_classification',
        'scope12_class',
        'scope3_cat11_class'
    ]
].head(20)

,company_name,near_term_target_classification,scope12_class,scope3_cat11_class
0,"""Hytex Plastic"" CJSC",1.5°C,1.5°C,NaN
2,& Co.,1.5°C,1.5°C,NaN
3,&Tradition,1.5°C,1.5°C,NaN
4,(ACIP) Alexandria Company for Industrial Packages,Well-below 2°C,Well-below 2°C,NaN
5,"10 to 2 Consulting, LLC",1.5°C,1.5°C,NaN
6,100 Percent Group Limited,1.5°C,1.5°C,NaN
8,2 Sisters Food Group,1.5°C,1.5°C,NaN
9,2-Connect,1.5°C,1.5°C,NaN
11,2050 Consulting,1.5°C,1.5°C,NaN
13,24 Ltd,1.5°C,1.5°C,NaN


In [25]:
# Scope 1·2 기준 목표 수준별 기업 수 확인
target_set['scope12_class'].value_counts(dropna=False)

scope12_class
1.5°C             11653
Well-below 2°C      323
2°C                  20
NaN                   1
Name: count, dtype: int64

In [26]:
# Scope 1·2 목표 수준별 비율 확인
scope12_pct = (
    target_set['scope12_class']
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(1)
)

scope12_pct

scope12_class
1.5°C             97.1
Well-below 2°C     2.7
2°C                0.2
NaN                0.0
Name: proportion, dtype: float64

### 분석 결과

SBTi 데이터에서 `Targets set` 상태인 기업 중 97.1%가 Scope 1·2 감축목표를 1.5°C 수준에 맞추고 있었다.

이를 통해 SBTi에서 목표를 설정한 기업들은 대부분 높은 수준의 감축목표를 설정하고 있다는 것을 확인할 수 있었다. 다만 이 데이터만으로 실제 감축이 얼마나 진행되고 있는지까지 판단할 수는 없다.

In [27]:
# Near-term 목표연도별 기업 수 확인
year_count = (
    target_set['near_term_year_clean']
    .value_counts()
    .sort_index()
)

year_count

near_term_year_clean
2020.0       2
2022.0       1
2024.0       4
2025.0      60
2026.0      35
2027.0      61
2028.0      83
2029.0     124
2030.0    9117
2031.0     318
2032.0     355
2033.0     359
2034.0     622
2035.0     832
2036.0      16
2037.0       2
2040.0       4
2050.0       1
Name: count, dtype: int64

In [28]:
# 목표연도가 너무 과거이거나 2040년 이후인 기업 확인
unusual_year = target_set[
    (target_set['near_term_year_clean'] <= 2025) |
    (target_set['near_term_year_clean'] >= 2040)
]

unusual_year[
    [
        'company_name',
        'near_term_target_year',
        'near_term_year_clean',
        'full_target_language'
    ]
]

,company_name,near_term_target_year,near_term_year_clean,full_target_language
88,AB InBev,2025.0,2025.0,Global Brewer AB InBev commits to reduce absol...
143,Accent Equity AB,2025.0,2025.0,Headline target: Accent Equity AB's portfolio ...
191,Actiam NV,2025.0,2025.0,Headline target: ACTIAM’s portfolio targets co...
259,"Advanced Micro Devices, Inc",2020.0,2020.0,Multinational semiconductor company AMD commit...
346,AIA Group Limited,2025.0,2025.0,Headline target: AIA Group's portfolio targets...
...,...,...,...,...
14476,Urban Partners,2025.0,2025.0,Headline target: \nUrban Partners' portfolio t...
14552,VALLOUREC,2025.0,2025.0,Vallourec commits to reduce absolute Scopes 1 ...
14652,"Veritas Technologies, LLC",FY2025,2025.0,Veritas commits to reduce absolute scope 1 and...
14709,Vidia GmbH,2025.0,2025.0,Headline target: Vidia GmbH’s portfolio target...


### 데이터 확인

Near-term 목표연도를 확인하는 과정에서 일부 기업의 연도값과 실제 목표 문구가 일치하지 않는 경우를 확인했다.
따라서 해당 컬럼을 그대로 사용한 목표연도 분석은 제외하고, 이후 필요할 경우 실제 목표 문구를 기준으로 다시 정리하기로 했다.

## 지역별 SBTi 기업 분포

SBTi 데이터에 등록된 기업들이 어느 지역에 많이 분포되어 있는지 확인했다.

In [29]:
# 지역별 기업 수 확인
region_count = df_sbti['region'].value_counts()

region_count

region
Europe                             7716
Asia                               5313
Northern America                   1860
Latin America and the Caribbean     347
Oceania                             227
Africa                              127
MENA                                 64
Name: count, dtype: int64

In [30]:
# 지역별 기업 비율 확인
region_pct = (
    df_sbti['region']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

region_pct

region
Europe                             49.3
Asia                               33.9
Northern America                   11.9
Latin America and the Caribbean     2.2
Oceania                             1.5
Africa                              0.8
MENA                                0.4
Name: proportion, dtype: float64

### 분석 결과

SBTi 데이터에 등록된 기업 중 유럽 기업이 49.3%로 가장 높은 비중을 차지했고, 아시아가 33.9%로 뒤를 이었다.

다만 이 결과는 전 세계 기업의 지역별 참여율을 의미하는 것이 아니라, 현재 SBTi 데이터에 포함된 기업들의 분포를 보여주는 결과이다. 따라서 유럽 기업이 다른 지역보다 ESG나 기후대응을 더 잘한다고 단정할 수는 없다.

In [31]:
# 지역별 기업 수와 비율을 그래프용 데이터로 정리
region_plot = pd.DataFrame({
    '지역': region_count.index,
    '기업수': region_count.values,
    '비율(%)': region_pct.values
})

# 지역별 SBTi 등록 기업 수 시각화
fig = px.bar(
    region_plot,
    x='기업수',
    y='지역',
    text='비율(%)',
    title='지역별 SBTi 등록 기업 분포'
)

# 막대 끝에 비율 표시
fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

# 기업 수가 많은 지역부터 위에 표시
fig.update_yaxes(autorange='reversed')

fig.show()

In [32]:
# 지역별 Near-term 목표 상태 비율 확인
region_status_pct = pd.crosstab(
    df_sbti['region'],
    df_sbti['near_term_status'],
    normalize='index'
).mul(100).round(1)

region_status_pct

near_term_status,Commitment removed,Committed,Targets set
region,,,
Africa,22.0,18.1,59.8
Asia,7.0,9.9,83.1
Europe,8.7,16.6,74.7
Latin America and the Caribbean,18.7,15.3,66.0
MENA,23.4,25.0,51.6
Northern America,13.9,15.5,70.6
Oceania,14.5,12.3,73.1


In [33]:
# 그래프를 만들기 위해 표 형태를 변환
region_status_plot = (
    region_status_pct
    .reset_index()
    .melt(
        id_vars='region',
        var_name='목표상태',
        value_name='비율'
    )
)

# 지역별 목표 상태 비율 시각화
fig = px.bar(
    region_status_plot,
    x='region',
    y='비율',
    color='목표상태',
    barmode='stack',
    title='지역별 Near-term 목표 상태 비율'
)

fig.update_layout(
    xaxis_title='지역',
    yaxis_title='비율(%)'
)

fig.show()

In [34]:
# 지역별 기업 수와 Targets set 비율을 함께 확인
region_compare = region_status_pct.copy()

region_compare['전체 기업수'] = df_sbti['region'].value_counts()

region_compare[
    ['전체 기업수', 'Targets set', 'Committed', 'Commitment removed']
]

near_term_status,전체 기업수,Targets set,Committed,Commitment removed
region,,,,
Africa,127,59.8,18.1,22.0
Asia,5313,83.1,9.9,7.0
Europe,7716,74.7,16.6,8.7
Latin America and the Caribbean,347,66.0,15.3,18.7
MENA,64,51.6,25.0,23.4
Northern America,1860,70.6,15.5,13.9
Oceania,227,73.1,12.3,14.5


### 분석 결과

지역별로 비교했을 때 아시아의 `Targets set` 비율이 83.1%로 가장 높았고, 유럽은 74.7%로 나타났다.

반면 MENA 지역은 `Targets set` 비율이 51.6%로 낮았고, `Committed`와 `Commitment removed` 비율은 각각 25.0%, 23.4%로 상대적으로 높았다.

다만 MENA는 전체 기업 수가 64개로 다른 지역보다 표본이 적기 때문에, 이 결과만으로 지역 전체의 기후대응 수준을 판단하기에는 한계가 있다.

## 산업별 SBTi 기업 분포

SBTi에 등록된 기업들이 어떤 산업에 많이 분포되어 있는지 확인했다.

In [35]:
# 산업별 기업 수 확인
sector_count = df_sbti['sector'].value_counts()

# 기업 수가 많은 상위 15개 산업 확인
sector_count.head(15)

sector
Professional Services                                                       1589
Electrical Equipment and Machinery                                          1264
Software and Services                                                       1100
Food and Beverage Processing                                                 990
Construction and Engineering                                                 974
Textiles, Apparel, Footwear and Luxury Goods                                 844
Trading Companies and Distributors, and Commercial Services and Supplies     763
Consumer Durables, Household and Personal Products                           627
Pharmaceuticals, Biotechnology and Life Sciences                             556
Automobiles and Components                                                   556
Technology Hardware and Equipment                                            466
Containers and Packaging                                                     450
Chemicals            

In [36]:
# 산업별 기업 수와 비율 계산
sector_table = pd.DataFrame({
    '기업수': df_sbti['sector'].value_counts(),
    '비율(%)': df_sbti['sector']
                .value_counts(normalize=True)
                .mul(100)
                .round(1)
})

# 상위 10개 산업 확인
sector_table.head(10)

,기업수,비율(%)
sector,,
Professional Services,1589,10.2
Electrical Equipment and Machinery,1264,8.1
Software and Services,1100,7.0
Food and Beverage Processing,990,6.3
Construction and Engineering,974,6.2
"Textiles, Apparel, Footwear and Luxury Goods",844,5.4
"Trading Companies and Distributors, and Commercial Services and Supplies",763,4.9
"Consumer Durables, Household and Personal Products",627,4.0
"Pharmaceuticals, Biotechnology and Life Sciences",556,3.6


In [37]:
# 기업 수가 많은 상위 10개 산업 이름 저장
top10_sectors = sector_table.head(10).index

# 상위 10개 산업 데이터만 추출
top10_df = df_sbti[
    df_sbti['sector'].isin(top10_sectors)
]

# 산업별 Near-term 목표 상태 비율 계산
sector_status_pct = pd.crosstab(
    top10_df['sector'],
    top10_df['near_term_status'],
    normalize='index'
).mul(100).round(1)

# 기존 기업 수 순서대로 정렬
sector_status_pct = sector_status_pct.loc[top10_sectors]

sector_status_pct

near_term_status,Commitment removed,Committed,Targets set
sector,,,
Professional Services,5.9,8.9,85.1
Electrical Equipment and Machinery,5.1,11.2,83.6
Software and Services,10.2,14.4,75.5
Food and Beverage Processing,9.1,23.8,67.1
Construction and Engineering,6.8,10.2,83.1
"Textiles, Apparel, Footwear and Luxury Goods",15.2,12.2,72.6
"Trading Companies and Distributors, and Commercial Services and Supplies",6.6,13.1,80.3
"Consumer Durables, Household and Personal Products",6.5,12.3,81.2
"Pharmaceuticals, Biotechnology and Life Sciences",7.4,25.9,66.7


In [38]:
# 상위 10개 산업의 목표 상태 비율을 그래프용 형태로 변환
sector_status_plot = (
    sector_status_pct
    .reset_index()
    .melt(
        id_vars='sector',
        var_name='목표상태',
        value_name='비율'
    )
)

# 산업별 Near-term 목표 상태 비율 시각화
fig = px.bar(
    sector_status_plot,
    x='sector',
    y='비율',
    color='목표상태',
    barmode='stack',
    title='산업별 Near-term 목표 상태 비율'
)

fig.update_layout(
    xaxis_title='산업',
    yaxis_title='비율(%)',
    xaxis_tickangle=-45
)

fig.show()

### 분석 결과

상위 10개 산업 모두 `Targets set` 비율이 가장 높게 나타났지만, 산업별로 목표 설정 단계에는 차이가 있었다.

특히 제약·바이오 산업은 `Committed` 비율이 25.9%, 식음료 산업은 23.8%로 다른 산업보다 높았다. 반면 섬유·의류·럭셔리 산업은 `Commitment removed` 비율이 15.2%로 가장 높게 나타났다.

따라서 SBTi에 등록된 기업이라도 산업에 따라 목표 설정 상태의 분포가 다르게 나타나는 것을 확인할 수 있었다.

## Net-zero 목표 현황

SBTi에 등록된 기업들이 Net-zero 목표를 어느 단계까지 설정하고 있는지 확인했다.

In [39]:
# Net-zero 목표 상태별 기업 수 확인
netzero_count = df_sbti['net_zero_status'].value_counts(dropna=False)

netzero_count

net_zero_status
NaN                   10599
Targets set            2805
Commitment removed     1211
Committed              1039
Name: count, dtype: int64

In [40]:
# Net-zero 목표 상태별 비율 확인
netzero_pct = (
    df_sbti['net_zero_status']
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(1)
)

netzero_pct

net_zero_status
NaN                   67.7
Targets set           17.9
Commitment removed     7.7
Committed              6.6
Name: proportion, dtype: float64

In [41]:
# Net-zero 상태와 Long-term 목표 상태를 같이 확인
pd.crosstab(
    df_sbti['net_zero_status'].fillna('미기재'),
    df_sbti['long_term_status'].fillna('미기재')
)

long_term_status,Targets set,미기재
net_zero_status,,
Commitment removed,0,1211
Committed,0,1039
Targets set,2752,53
미기재,0,10599


In [42]:
# Long-term과 Net-zero 상태의 실제 값 다시 확인
print('Long-term status')
print(df_sbti['long_term_status'].value_counts(dropna=False))

print('\nNet-zero status')
print(df_sbti['net_zero_status'].value_counts(dropna=False))

Long-term status
long_term_status
NaN            12902
Targets set     2752
Name: count, dtype: int64

Net-zero status
net_zero_status
NaN                   10599
Targets set            2805
Commitment removed     1211
Committed              1039
Name: count, dtype: int64


In [43]:
# Net-zero는 Targets set인데 Long-term 상태가 비어 있는 기업 확인
netzero_only = df_sbti[
    (df_sbti['net_zero_status'] == 'Targets set') &
    (df_sbti['long_term_status'].isna())
]

# 몇 개인지 확인
netzero_only.shape

(53, 23)

In [44]:
# 기업 유형 확인
netzero_only['organization_type'].value_counts()

organization_type
SME          52
Corporate     1
Name: count, dtype: int64

In [45]:
# Net-zero는 설정되어 있지만 Long-term 상태가 비어 있는 기업 확인
netzero_only[
    [
        'company_name',
        'organization_type',
        'net_zero_year',
        'long_term_target_year',
        'full_target_language'
    ]
].head(10)

,company_name,organization_type,net_zero_year,long_term_target_year,full_target_language
332,Agro de Souza S.A.,SME,2034.0,NaN,This target was approved using a streamlined t...
492,AlivaMab Biologics LLC,SME,2035.0,NaN,This target was approved using a streamlined t...
674,AMPECO,SME,2030.0,NaN,This target was approved using a streamlined t...
679,AMR LLP,SME,2030.0,NaN,This target was approved using a streamlined t...
1407,BDO Al-Amri,SME,FY2050,NaN,This target was approved using a streamlined t...
1515,Beijing Bofu Medical Consulting Co Ltd,SME,2030.0,NaN,This target was approved using a streamlined t...
1547,BELFOR (UK) Limited,SME,2030.0,NaN,This target was approved using a streamlined t...
1595,Berkeley Capital Group (BCG),SME,2028.0,NaN,This target was approved using a streamlined t...
2085,BW Workplace Experts,SME,2030.0,NaN,This target was approved using a streamlined t...
2711,CM International (CMI),SME,2030.0,NaN,This target was approved using a streamlined t...


In [46]:
# Net-zero 상태가 비어 있는 기업만 추출
netzero_missing = df_sbti[
    df_sbti['net_zero_status'].isna()
].copy()

# 목표 문구에 net-zero가 포함되어 있는지 확인
netzero_missing['netzero_text'] = (
    netzero_missing['full_target_language']
    .fillna('')
    .str.contains('net-zero', case=False)
)

# 결과 확인
netzero_missing['netzero_text'].value_counts()

netzero_text
False    10598
True         1
Name: count, dtype: int64

In [47]:
# net-zero / net zero / netzero 표현을 모두 확인
netzero_missing['netzero_text_check'] = (
    netzero_missing['full_target_language']
    .fillna('')
    .str.contains(
        r'net[\s\-–—]?zero',
        case=False,
        regex=True
    )
)

netzero_missing['netzero_text_check'].value_counts()

netzero_text_check
False    10597
True         2
Name: count, dtype: int64

In [48]:
# Net-zero 상태는 비어 있는데
# 목표 문구에는 net-zero 표현이 있는 2개 기업 확인
netzero_text_exception = netzero_missing[
    netzero_missing['netzero_text_check'] == True
]

netzero_text_exception[
    [
        'company_name',
        'organization_type',
        'near_term_status',
        'long_term_status',
        'net_zero_status',
        'full_target_language'
    ]
]

,company_name,organization_type,near_term_status,long_term_status,net_zero_status,full_target_language
1207,Avia Pharma AB,Corporate,Targets set,NaN,NaN,Overall Net-Zero Target:\nAvia Pharma AB commi...
9454,Net Zero Group,SME,Targets set,NaN,NaN,This target was approved using a streamlined t...


### 분석 결과

SBTi에 등록된 기업 중 17.9%가 Net-zero 목표를 설정한 상태였고, 6.6%는 목표 설정을 약속한 단계였다. 반면 67.7%는 Net-zero 상태가 별도로 등록되어 있지 않았다.

Net-zero 상태가 비어 있는 기업의 목표 문구도 추가로 확인한 결과, 대부분 명시적인 Net-zero 목표가 확인되지 않았다. 다만 Avia Pharma AB처럼 목표 문구에는 Net-zero 목표가 있지만 상태 컬럼에는 반영되지 않은 예외 사례도 확인되었다.

따라서 이번 분석에서는 원본 상태값을 그대로 사용하되, 일부 데이터 누락 가능성이 있다는 점을 함께 고려했다.

In [49]:
# Net-zero 목표 상태별 비율 정리
netzero_plot = pd.DataFrame({
    '상태': ['Targets set', 'Committed', 'Commitment removed', '미기재'],
    '비율': [17.9, 6.6, 7.7, 67.7]
})

# 그래프 생성
fig = px.bar(
    netzero_plot,
    x='상태',
    y='비율',
    text='비율',
    title='SBTi 기업의 Net-zero 목표 상태'
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='Net-zero 목표 상태',
    yaxis_title='비율(%)'
)

fig.show()

## Near-term 목표와 Net-zero 목표 비교

단기 감축목표와 Net-zero 목표의 설정 수준에 차이가 있는지 비교했다.

In [50]:
# 비교할 목표 상태 순서
status_order = [
    'Targets set',
    'Committed',
    'Commitment removed',
    '미기재'
]

# Near-term 목표 상태 비율
near_compare = (
    df_sbti['near_term_status']
    .fillna('미기재')
    .value_counts(normalize=True)
    .mul(100)
    .reindex(status_order, fill_value=0)
    .round(1)
)

# Net-zero 목표 상태 비율
netzero_compare = (
    df_sbti['net_zero_status']
    .fillna('미기재')
    .value_counts(normalize=True)
    .mul(100)
    .reindex(status_order, fill_value=0)
    .round(1)
)

# 비교표 만들기
target_compare = pd.DataFrame({
    'Near-term': near_compare,
    'Net-zero': netzero_compare
})

target_compare

,Near-term,Net-zero
Targets set,76.6,17.9
Committed,14.1,6.6
Commitment removed,9.2,7.7
미기재,0.0,67.7


In [51]:
# 비교표를 그래프용 형태로 변환
target_compare_plot = (
    target_compare
    .reset_index()
    .rename(columns={'index': '목표상태'})
    .melt(
        id_vars='목표상태',
        var_name='목표유형',
        value_name='비율'
    )
)

# Near-term과 Net-zero 목표 상태 비교
fig = px.bar(
    target_compare_plot,
    x='목표상태',
    y='비율',
    color='목표유형',
    barmode='group',
    text='비율',
    title='Near-term과 Net-zero 목표 상태 비교'
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='목표 상태',
    yaxis_title='비율(%)'
)

fig.show()

### 분석 결과

Near-term과 Net-zero 목표 상태를 비교한 결과, `Targets set` 비율에서 가장 큰 차이가 나타났다.
Near-term은 76.6%가 목표를 설정한 상태였지만, Net-zero는 17.9%로 58.7%p 차이가 났다.

반대로 Net-zero는 상태가 미기재된 기업이 67.7%로 높게 나타났으며, Near-term에서는 거의 확인되지 않았다.

`Committed`는 7.5%p 차이가 있었지만, `Commitment removed`는 1.5%p 차이로 비교적 비슷하게 나타났다.

이를 통해 SBTi 등록 기업들은 단기 감축목표 설정은 상당히 진행되어 있지만, Net-zero 목표 설정까지 이어진 기업의 비중은 상대적으로 낮다는 것을 확인할 수 있었다.

## 한국 기업 SBTi 현황

글로벌 SBTi 데이터에서 한국 기업을 따로 추출해 전체 기업과 목표 설정 현황을 비교해보려고 한다.

In [52]:
# location 컬럼에서 Korea가 들어간 값 확인
df_sbti[
    df_sbti['location']
    .str.contains('Korea', case=False, na=False)
]['location'].value_counts()

location
Korea, Republic of    125
Name: count, dtype: int64

In [53]:
# 한국 기업만 추출
korea_sbti = df_sbti[
    df_sbti['location'] == 'Korea, Republic of'
].copy()

# 한국 기업 수 확인
korea_sbti.shape

(125, 23)

In [54]:
# 한국 기업의 Near-term 목표 상태 비율 확인
korea_near_status = (
    korea_sbti['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

korea_near_status

near_term_status
Targets set           72.0
Committed             16.0
Commitment removed    12.0
Name: proportion, dtype: float64

In [55]:
# 전체 SBTi 기업의 Near-term 목표 상태 비율
global_near_status = (
    df_sbti['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 전체와 한국 비교표 만들기
near_compare_korea = pd.DataFrame({
    '전체 SBTi': global_near_status,
    '한국': korea_near_status
})

near_compare_koreax

,전체 SBTi,한국
near_term_status,,
Targets set,76.6,72.0
Committed,14.1,16.0
Commitment removed,9.2,12.0


In [56]:
# 비교표를 그래프용 형태로 변환
near_compare_plot = (
    near_compare_korea
    .reset_index()
    .melt(
        id_vars='near_term_status',
        var_name='구분',
        value_name='비율'
    )
)

# 전체 SBTi와 한국 기업 비교
fig = px.bar(
    near_compare_plot,
    x='near_term_status',
    y='비율',
    color='구분',
    barmode='group',
    text='비율',
    title='전체 SBTi와 한국 기업의 Near-term 목표 상태 비교'
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='목표 상태',
    yaxis_title='비율(%)'
)

fig.show()

### 분석 결과

한국 기업의 Near-term 목표 상태를 전체 SBTi 기업과 비교한 결과, `Targets set` 비율은 한국이 72.0%로 전체 SBTi의 76.6%보다 4.6%p 낮게 나타났다.

반면 `Committed`는 한국이 16.0%로 전체보다 1.9%p 높았고, `Commitment removed`도 한국이 12.0%로 전체보다 2.8%p 높았다.

전체적으로 보면 한국 기업도 대부분 Near-term 목표를 설정한 상태였지만, 전체 SBTi 기업과 비교하면 목표 설정 완료 비율은 조금 낮고 아직 목표 설정 단계이거나 철회된 비율은 조금 높게 나타났다.

다만 한국 기업은 125개로 전체 SBTi 기업보다 표본 수가 적고 산업 구성도 다를 수 있기 때문에, 이 차이만으로 한국 기업의 기후대응 수준이 낮다고 단정하기는 어렵다.


## 한국 기업의 Net-zero 목표 비교

한국 기업의 Net-zero 목표 상태가 전체 SBTi 기업과 어떤 차이가 있는지 비교했다.

In [57]:
# 한국 기업의 Net-zero 목표 상태 비율
korea_netzero_status = (
    korea_sbti['net_zero_status']
    .fillna('미기재')
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

korea_netzero_status

net_zero_status
미기재                   67.2
Targets set           14.4
Commitment removed     9.6
Committed              8.8
Name: proportion, dtype: float64

In [58]:
# 전체 SBTi의 Net-zero 목표 상태 비율
global_netzero_status = (
    df_sbti['net_zero_status']
    .fillna('미기재')
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 비교표 만들기
netzero_compare_korea = pd.DataFrame({
    '전체 SBTi': global_netzero_status,
    '한국': korea_netzero_status
})

netzero_compare_korea

,전체 SBTi,한국
net_zero_status,,
미기재,67.7,67.2
Targets set,17.9,14.4
Commitment removed,7.7,9.6
Committed,6.6,8.8


In [59]:
# 비교표를 그래프용 형태로 변환
netzero_compare_plot = (
    netzero_compare_korea
    .reset_index()
    .melt(
        id_vars='net_zero_status',
        var_name='구분',
        value_name='비율'
    )
)

# 전체 SBTi와 한국 기업의 Net-zero 목표 상태 비교
fig = px.bar(
    netzero_compare_plot,
    x='net_zero_status',
    y='비율',
    color='구분',
    barmode='group',
    text='비율',
    title='전체 SBTi와 한국 기업의 Net-zero 목표 상태 비교'
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='Net-zero 목표 상태',
    yaxis_title='비율(%)'
)

fig.show()

### 분석 결과

한국 기업의 Net-zero 목표 상태를 전체 SBTi 기업과 비교한 결과, 전체적인 분포는 비슷하게 나타났다.

Net-zero 상태가 미기재된 비율은 전체 SBTi가 67.7%, 한국이 67.2%로 거의 차이가 없었다. 반면 `Targets set` 비율은 한국이 14.4%로 전체 SBTi의 17.9%보다 3.5%p 낮았다.

`Committed`는 한국이 8.8%로 전체보다 2.2%p 높았고, `Commitment removed`도 9.6%로 전체보다 1.9%p 높게 나타났다.

따라서 한국 기업의 Net-zero 목표 상태는 전체 SBTi 기업과 큰 차이는 없었지만, 목표 설정이 완료된 비율은 조금 낮고 약속 단계이거나 철회된 비율은 조금 높게 나타났다.

다만 한국 기업은 125개로 표본 수가 적고 산업 구성 차이도 있기 때문에, 이 결과만으로 한국 기업의 Net-zero 대응 수준을 전체보다 낮다고 판단하기는 어렵다.

## 한국 기업의 산업 분포

한국 SBTi 기업이 어떤 산업에 많이 분포되어 있는지 확인하고, 전체 SBTi 기업의 산업 분포와 차이가 있는지 살펴봤다.

In [60]:
# 한국 기업의 산업별 기업 수 확인
korea_sector_count = korea_sbti['sector'].value_counts()

# 상위 10개 산업 확인
korea_sector_count.head(10)

sector
Automobiles and Components                                                  18
Banks, Diverse Financials, Insurance                                        13
Professional Services                                                       12
Textiles, Apparel, Footwear and Luxury Goods                                12
Technology Hardware and Equipment                                            9
Trading Companies and Distributors, and Commercial Services and Supplies     6
Software and Services                                                        5
Consumer Durables, Household and Personal Products                           5
Chemicals                                                                    5
Semiconductors and Semiconductors Equipment                                  5
Name: count, dtype: int64

In [61]:
# 한국에서 기업 수가 많은 상위 10개 산업
korea_top10 = korea_sector_count.head(10).index

# 한국 상위 10개 산업의 비율
korea_sector_pct = (
    korea_sbti['sector']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 같은 산업이 전체 SBTi에서는 어느 정도 비중인지 계산
global_sector_pct = (
    df_sbti['sector']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 한국 상위 10개 산업을 기준으로 비교표 만들기
sector_compare_korea = pd.DataFrame({
    '전체 SBTi': global_sector_pct.reindex(korea_top10),
    '한국': korea_sector_pct.reindex(korea_top10)
})

sector_compare_korea

,전체 SBTi,한국
sector,,
Automobiles and Components,3.6,14.4
"Banks, Diverse Financials, Insurance",2.1,10.4
Professional Services,10.2,9.6
"Textiles, Apparel, Footwear and Luxury Goods",5.4,9.6
Technology Hardware and Equipment,3.0,7.2
"Trading Companies and Distributors, and Commercial Services and Supplies",4.9,4.8
Software and Services,7.0,4.0
"Consumer Durables, Household and Personal Products",4.0,4.0
Chemicals,2.8,4.0


### 분석 결과

한국 SBTi 기업의 산업 분포를 전체 SBTi 기업과 비교한 결과, 산업 구성에 차이가 나타났다.

한국은 자동차·부품 산업이 14.4%로 전체 SBTi의 3.6%보다 10.8%p 높았고, 금융도 10.4%로 전체의 2.1%보다 높은 비중을 차지했다. 기술 하드웨어와 반도체 산업도 전체 SBTi보다 한국에서 상대적으로 높은 비중을 보였다.

반면 소프트웨어 산업은 한국이 4.0%로 전체 SBTi의 7.0%보다 낮았으며, 전문서비스는 두 집단에서 비슷한 수준으로 나타났다.

따라서 앞에서 확인한 한국과 전체 SBTi의 목표 상태 차이는 국가 차이뿐 아니라 산업 구성의 차이에도 영향을 받을 수 있다.

## 같은 산업 내 한국 기업 비교

한국과 전체 SBTi 기업의 산업 구성이 다르기 때문에, 단순 국가 비교만으로는 차이를 해석하기 어렵다.
먼저 한국에서 비중이 가장 높은 자동차·부품 산업을 대상으로 같은 산업 안에서 Near-term 목표 상태를 비교했다.

In [62]:
# 자동차·부품 산업 데이터 추출
auto_df = df_sbti[
    df_sbti['sector'] == 'Automobiles and Components'
].copy()

# 전체 자동차 기업 수와 한국 자동차 기업 수 확인
print('전체 자동차·부품 기업 수:', len(auto_df))
print(
    '한국 자동차·부품 기업 수:',
    len(auto_df[auto_df['location'] == 'Korea, Republic of'])
)

전체 자동차·부품 기업 수: 556
한국 자동차·부품 기업 수: 18


In [63]:
# 전체 자동차·부품 기업의 Near-term 목표 상태 비율
auto_global_status = (
    auto_df['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 한국 자동차·부품 기업만 추출
auto_korea = auto_df[
    auto_df['location'] == 'Korea, Republic of'
]

# 한국 자동차·부품 기업의 Near-term 목표 상태 비율
auto_korea_status = (
    auto_korea['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 비교표 만들기
auto_status_compare = pd.DataFrame({
    '전체 자동차·부품': auto_global_status,
    '한국 자동차·부품': auto_korea_status
})

auto_status_compare

,전체 자동차·부품,한국 자동차·부품
near_term_status,,
Targets set,73.0,55.6
Committed,16.0,22.2
Commitment removed,11.0,22.2


In [64]:
# 한국을 제외한 자동차·부품 기업
auto_non_korea = auto_df[
    auto_df['location'] != 'Korea, Republic of'
]

# 한국 제외 자동차·부품 기업의 Near-term 상태 비율
auto_non_korea_status = (
    auto_non_korea['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 한국 자동차·부품 기업과 비교
auto_compare_clean = pd.DataFrame({
    '한국 제외 자동차·부품': auto_non_korea_status,
    '한국 자동차·부품': auto_korea_status
})

auto_compare_clean

,한국 제외 자동차·부품,한국 자동차·부품
near_term_status,,
Targets set,73.6,55.6
Committed,15.8,22.2
Commitment removed,10.6,22.2


### 분석 결과

산업 구성 차이의 영향을 줄이기 위해 자동차·부품 산업만 따로 비교했다.

한국을 제외한 자동차·부품 기업의 `Targets set` 비율은 73.6%였지만, 한국 자동차·부품 기업은 55.6%로 18.0%p 낮게 나타났다.

반면 한국 기업의 `Committed` 비율은 22.2%로 한국 제외 기업보다 6.4%p 높았고, `Commitment removed`도 22.2%로 11.6%p 높았다.

따라서 전체 산업 구성이 다른 점을 고려한 뒤에도 한국 자동차·부품 기업 표본에서는 목표 상태의 차이가 나타났다. 다만 한국 자동차·부품 기업은 18개로 표본이 적기 때문에 이 결과를 한국 자동차 산업 전체의 특성으로 확대해서 해석하지는 않았다.

## 금융 산업 내 한국 기업 비교

자동차·부품 산업에서 확인한 차이가 다른 산업에서도 나타나는지 확인하기 위해, 한국에서 두 번째로 기업 수가 많은 금융 산업을 추가로 비교했다.

In [65]:
# 금융 산업 데이터 추출
finance_df = df_sbti[
    df_sbti['sector'] == 'Banks, Diverse Financials, Insurance'
].copy()

# 한국 금융 기업
finance_korea = finance_df[
    finance_df['location'] == 'Korea, Republic of'
]

# 한국을 제외한 금융 기업
finance_non_korea = finance_df[
    finance_df['location'] != 'Korea, Republic of'
]

# 표본 수 확인
print('한국 제외 금융 기업 수:', len(finance_non_korea))
print('한국 금융 기업 수:', len(finance_korea))

한국 제외 금융 기업 수: 310
한국 금융 기업 수: 13


In [66]:
# 한국 제외 금융 기업의 Near-term 목표 상태 비율
finance_non_korea_status = (
    finance_non_korea['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 한국 금융 기업의 Near-term 목표 상태 비율
finance_korea_status = (
    finance_korea['near_term_status']
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

# 비교표 만들기
finance_compare = pd.DataFrame({
    '한국 제외 금융': finance_non_korea_status,
    '한국 금융': finance_korea_status
})

finance_compare

,한국 제외 금융,한국 금융
near_term_status,,
Commitment removed,17.1,NaN
Committed,23.2,NaN
Targets set,59.7,100.0


### 분석 결과

금융 산업에서는 자동차·부품 산업과 다른 결과가 나타났다.

한국 금융 기업 13개는 모두 `Targets set` 상태로 나타난 반면, 한국을 제외한 금융 기업의 `Targets set` 비율은 59.7%였다. 한국 제외 금융 기업은 `Committed`가 23.2%, `Commitment removed`가 17.1%를 차지했다.

자동차·부품 산업에서는 한국 기업의 `Targets set` 비율이 상대적으로 낮았지만, 금융 산업에서는 오히려 한국 기업이 높게 나타났다.

따라서 한국 기업 전체의 목표 설정 비율 차이를 특정 국가의 특성만으로 설명하기는 어렵고, 산업별로 서로 다른 패턴이 나타날 수 있다는 것을 확인했다. 다만 한국 금융 기업은 13개로 표본이 적기 때문에 금융 산업 전체의 특성으로 확대해서 해석하지는 않았다.

## 산업별 한국 기업의 Near-term 목표 비교

자동차와 금융 산업에서 서로 다른 결과가 나타났기 때문에, 한국 기업이 5개 이상 포함된 산업을 대상으로 `Targets set` 비율을 전체적으로 비교했다.

In [67]:
# 한국 기업이 산업별로 몇 개 있는지 확인
korea_sector_n = (
    korea_sbti['sector']
    .value_counts()
)

# 한국 기업이 5개 이상 있는 산업만 선택
compare_sectors = korea_sector_n[
    korea_sector_n >= 5
].index

results = []

for sector in compare_sectors:

    # 해당 산업의 한국 기업
    kr = df_sbti[
        (df_sbti['sector'] == sector) &
        (df_sbti['location'] == 'Korea, Republic of')
    ]

    # 해당 산업의 한국 제외 기업
    non_kr = df_sbti[
        (df_sbti['sector'] == sector) &
        (df_sbti['location'] != 'Korea, Republic of')
    ]

    # Targets set 비율 계산
    kr_target_pct = (
        (kr['near_term_status'] == 'Targets set')
        .mean() * 100
    )

    non_kr_target_pct = (
        (non_kr['near_term_status'] == 'Targets set')
        .mean() * 100
    )

    results.append({
        '산업': sector,
        '한국 기업수': len(kr),
        '한국 제외 기업수': len(non_kr),
        '한국 Targets set(%)': round(kr_target_pct, 1),
        '한국 제외 Targets set(%)': round(non_kr_target_pct, 1),
        '차이(%p)': round(kr_target_pct - non_kr_target_pct, 1)
    })

# 비교표 생성
sector_target_compare = pd.DataFrame(results)

# 차이가 큰 순서대로 정렬
sector_target_compare = sector_target_compare.sort_values(
    '차이(%p)',
    ascending=False
)

sector_target_compare

,산업,한국 기업수,한국 제외 기업수,한국 Targets set(%),한국 제외 Targets set(%),차이(%p)
1,"Banks, Diverse Financials, Insurance",13,310,100.0,59.7,40.3
5,"Trading Companies and Distributors, and Commer...",6,757,100.0,80.2,19.8
7,"Consumer Durables, Household and Personal Prod...",5,622,80.0,81.2,-1.2
2,Professional Services,12,1577,75.0,85.2,-10.2
8,Chemicals,5,438,60.0,70.3,-10.3
3,"Textiles, Apparel, Footwear and Luxury Goods",12,832,58.3,72.8,-14.5
6,Software and Services,5,1095,60.0,75.5,-15.5
0,Automobiles and Components,18,538,55.6,73.6,-18.1
4,Technology Hardware and Equipment,9,457,55.6,74.0,-18.4
9,Semiconductors and Semiconductors Equipment,5,158,0.0,67.7,-67.7


### 분석 결과

한국 기업이 5개 이상 포함된 산업을 대상으로 `Targets set` 비율을 비교한 결과, 산업별로 서로 다른 패턴이 나타났다.

금융 산업은 한국 기업의 `Targets set` 비율이 100.0%로 한국 제외 기업보다 40.3%p 높았고, 유통·상사업도 한국 기업이 19.8%p 높게 나타났다.

반면 자동차·부품, 기술 하드웨어, 소프트웨어 등에서는 한국 기업의 `Targets set` 비율이 상대적으로 낮았다.

따라서 한국 기업의 Near-term 목표 상태는 국가 차이만으로 설명하기 어렵고, 산업별 특성에 따라 차이가 크게 나타나는 것을 확인했다.

다만 일부 산업은 한국 기업 수가 5~10개 수준으로 적어 비율 변동이 클 수 있기 때문에, 개별 산업 결과를 전체 산업의 특성으로 확대해서 해석하지는 않았다.

## 산업 구성을 고려한 한국 기업 비교

한국과 전체 SBTi 기업은 산업 구성이 다르기 때문에, 산업 구성 차이를 고려한 비교를 추가로 진행했다.

한국 기업의 실제 산업 비중은 그대로 유지한 상태에서, 각 산업이 한국 제외 기업과 같은 `Targets set` 비율을 보였다고 가정했을 때의 기대 비율을 계산했다.

In [68]:
# 한국 기업의 산업별 기업 수
korea_sector_size = (
    korea_sbti['sector']
    .value_counts()
)

# 한국 기업의 산업별 비중
korea_sector_weight = (
    korea_sector_size / len(korea_sbti)
)

# 산업별 한국 제외 기업의 Targets set 비율 계산
non_korea_sector_target = (
    df_sbti[
        df_sbti['location'] != 'Korea, Republic of'
    ]
    .groupby('sector')['near_term_status']
    .apply(lambda x: (x == 'Targets set').mean())
)

# 한국의 산업 비중과 한국 제외 기업의 산업별 Targets set 비율 결합
adjust_table = pd.DataFrame({
    '한국 산업 비중': korea_sector_weight,
    '한국 제외 Targets set 비율': non_korea_sector_target
}).dropna()

# 산업 구성을 고려한 기대 Targets set 비율 계산
expected_target_rate = (
    adjust_table['한국 산업 비중']
    * adjust_table['한국 제외 Targets set 비율']
).sum() * 100

# 한국 실제 Targets set 비율
actual_target_rate = (
    (korea_sbti['near_term_status'] == 'Targets set')
    .mean() * 100
)

print('한국 실제 Targets set 비율:', round(actual_target_rate, 1), '%')
print('산업 구성 보정 기대 비율:', round(expected_target_rate, 1), '%')
print('차이:', round(actual_target_rate - expected_target_rate, 1), '%p')

한국 실제 Targets set 비율: 72.0 %
산업 구성 보정 기대 비율: 75.1 %
차이: -3.1 %p


### 분석 결과

한국 기업의 실제 `Targets set` 비율은 72.0%였고, 한국의 산업 구성을 유지한 상태에서 각 산업이 한국 제외 기업과 같은 목표 설정 비율을 보였다고 가정했을 때의 기대 비율은 75.1%로 나타났다.

산업 구성을 고려하기 전에는 전체 SBTi 기업과 한국 기업의 `Targets set` 비율 차이가 4.6%p였지만, 산업 구성을 고려한 비교에서는 차이가 3.1%p로 줄어들었다.

이를 통해 한국과 전체 SBTi 기업의 목표 설정 비율 차이 중 일부는 산업 구성 차이와 관련되어 있을 가능성을 확인했다. 다만 산업 구성을 고려한 이후에도 차이가 남아 있어 산업 구성만으로 전체 차이를 설명하기는 어려웠다.

이번 비교는 산업만을 기준으로 보정한 결과이기 때문에 기업 규모나 지역, 기업 유형 등 다른 요인의 영향까지 설명하는 것은 아니다.

In [69]:
# 보정에 사용된 한국 산업 비중의 합이 100%인지 확인
adjust_table['한국 산업 비중'].sum()

np.float64(1.0)

## SBTi 분석 정리

SBTi 등록 기업을 분석한 결과, Near-term 목표는 76.6%가 `Targets set` 상태였지만 Net-zero 목표는 17.9%로 큰 차이가 나타났다.

지역과 산업별로도 목표 상태의 차이가 있었으며, 한국 기업은 전체 SBTi 기업보다 Near-term `Targets set` 비율이 4.6%p 낮았다.

하지만 한국과 전체 SBTi 기업의 산업 구성이 달랐고, 이를 고려해 비교한 결과 한국의 실제 `Targets set` 비율은 72.0%, 산업 구성 보정 기대 비율은 75.1%로 차이가 3.1%p까지 줄어들었다.

따라서 한국과 전체 SBTi 기업의 차이 중 일부는 산업 구성과 관련되어 있을 가능성이 있었지만, 산업 구성만으로 전체 차이를 설명할 수는 없었다.